# SONICS – dataset check

This notebook checks whether the SONICS files on disk match what is written in the documentation: the main paper (Rahman et al., arXiv version 4), the Hugging Face dataset card `README.md` and the dataset pages. All expected values and their sources are collected in `draft_notes_files/05_sonics/05_sonics_detailed.md` and `docs/05_sonics.md`.

**Important:** SONICS distributes only the **fake songs** as audio. The real songs are given only as metadata (`real_songs.csv`, with a YouTube ID). Checks on real songs therefore use the metadata only.

**How to use it**

- Run the cells from top to bottom.
- The notebook only **reads** the data – nothing in `datasets/` is changed. It reads only the headers of the audio files (`soundfile.info`), not the sound itself.
- Every check prints one line:
  - `[OK]` – the files match the documentation,
  - `[DIFF]` – the files do not match the documentation,
  - `[INFO]` – an observation, there is nothing to compare with.

**Output files**

In `outputs/05_sonics_dataset_exploration_output/` (not committed to git):

- `audio_info.csv` – properties of every audio file (step 8a),
- `checks.csv` – table of all checks (step 9).

In `results/05_sonics/05_sonics_dataset_exploration_output/` (committed to git):

- `dataset_exploration_report.md` – report with the results of all steps (step 10).

### Setup – paths and settings

**What it does:** Loads the libraries, finds the project folder and sets the paths to the dataset (`datasets/05_sonics`), to the outputs folder for working files and to the results folder for the report. The audio of the fake songs comes in ten archives `fake_songs/part_01.zip` … `part_10.zip`; the `.mp3` files are searched in all subfolders of `fake_songs/`, so the notebook does not depend on the names of the folders the archives were unpacked into. It also defines the column lists of the dataset card, the label values, the generators and Table 3 of the paper.

**Why:** All the next cells use these paths and values. The paths are relative to the project folder, so the notebook works on any computer without changes. The names of the generators in the files (`chirp-v3.5`, `udio-120s`, …) differ from the paper (Suno v3.5, Udio 130, …); the mapping comes from the detailed note (section 7) – it is an inference, because no source states it, but the counts per generator match Table 3 of the paper.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("pyproject.toml not found")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "datasets" / "05_sonics"
AUDIO_DIR = DATA_DIR / "fake_songs"
FAKE_CSV = DATA_DIR / "fake_songs.csv"
REAL_CSV = DATA_DIR / "real_songs.csv"
METADATA_JSON = DATA_DIR / "metadata.json"
SPLITS = ["train", "valid", "test"]
NOTEBOOK_NAME = "05_sonics_dataset_exploration"
OUTPUTS_DIR = REPO_ROOT / "outputs" / f"{NOTEBOOK_NAME}_output"
RESULTS_DIR = REPO_ROOT / "results" / "05_sonics" / f"{NOTEBOOK_NAME}_output"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# short names of the sources
PAPER = "main paper (arXiv v4)"
CARD = "README.md (Hugging Face dataset card)"
WEB = "Hugging Face page (web only)"
NOTE = "detailed note (not documented by the authors)"
NOT_STATED = "not stated (observation)"

# column tables of README.md ("Metadata Properties")
CARD_FAKE_COLUMNS = [
    "id", "filename", "title", "duration", "algorithm", "style", "source",
    "lyrics_features", "topic", "genre", "mood", "label", "target", "split",
]  # fmt: skip
CARD_REAL_COLUMNS = [
    "id", "filename", "title", "artist", "year", "lyrics", "lyrics_features", "duration",
    "youtube_id", "label", "artist_overlap", "target", "skip_time", "no_vocal", "split",
]  # fmt: skip
FAKE_TYPES = ["full fake", "half fake", "mostly fake"]

# generator names in the files -> names in the paper (detailed note, section 7; inference)
GENERATORS = {
    "chirp-v2-xxl-alpha": "Suno v2",
    "chirp-v3": "Suno v3",
    "chirp-v3.5": "Suno v3.5",
    "udio-30s": "Udio 32",
    "udio-120s": "Udio 130",
}
# maximum song length of each generator in seconds (main paper, p. 4, section 3.1.2)
GENERATOR_LIMITS = {"Suno v2": 80, "Suno v3": 120, "Suno v3.5": 240, "Udio 32": 32, "Udio 130": 130}
UNSEEN = {"Suno v2", "Suno v3", "Udio 32"}  # main paper, p. 7, section 4.1

# Table 3 of the main paper (p. 4): songs per fake type, generator and split
TABLE3 = pd.DataFrame(
    [
        ("full fake", "Suno v3.5", 774, 9, 77),
        ("full fake", "Udio 130", 656, 12, 53),
        ("full fake", "Suno v3", 0, 37, 206),
        ("full fake", "Suno v2", 0, 16, 100),
        ("full fake", "Udio 32", 0, 30, 203),
        ("half fake", "Suno v3.5", 4558, 67, 396),
        ("half fake", "Suno v3", 0, 117, 634),
        ("half fake", "Suno v2", 0, 51, 309),
        ("mostly fake", "Suno v3.5", 11863, 196, 1117),
        ("mostly fake", "Udio 130", 16172, 286, 1566),
        ("mostly fake", "Suno v3", 0, 494, 2797),
        ("mostly fake", "Suno v2", 0, 258, 1350),
        ("mostly fake", "Udio 32", 0, 700, 3970),
    ],
    columns=["label", "generator", *SPLITS],
).set_index(["label", "generator"])
TABLE3_SPLIT_TOTALS = {"train": 66709, "valid": 4440, "test": 26015}
TABLE3_REAL = {"train": 32686, "valid": 2167, "test": 13237}

print("Data folder:", DATA_DIR.relative_to(REPO_ROOT), "| exists:", DATA_DIR.exists())
print("Audio folder:", AUDIO_DIR.relative_to(REPO_ROOT), "| exists:", AUDIO_DIR.exists())

### Check list – helper function

**What it does:** Defines `add_check`, a small function that compares an **expected** value (from the documentation) with a **measured** value (from the files on disk). It prints `[OK]` if they match, `[DIFF]` if they do not, and `[INFO]` if there is nothing to compare with (only an observation). Every result is saved in the list `checks`.

**Why:** All results are collected in one place, so at the end they can be shown in one table and written to the report.

In [ ]:
checks = []


def add_check(check: str, source: str, expected, measured) -> None:
    ok = None if expected is None else expected == measured
    checks.append(
        {
            "check": check,
            "source": source,
            "expected": expected,
            "measured": measured,
            "ok": ok,
        }
    )
    status = "INFO" if ok is None else ("OK" if ok else "DIFF")
    print(f"[{status}] {check}: expected={expected!r}, measured={measured!r}")

### Step 1 – Folder structure

**What it does:** Checks that the items of the "Folder Structure" in `README.md` exist (`fake_songs/`, `real_songs.csv`, `fake_songs.csv`), that the data folder contains exactly the files listed on the Hugging Face "Files and versions" page, that `fake_songs/` contains the ten archives `part_01.zip` … `part_10.zip` (Hugging Face folder page), and that the audio files are `.mp3` files (`README.md`: `fake_songs/yyy.mp3`). Hidden files (for example `.DS_Store`, `.cache`) are ignored.

**Why:** Confirms that everything was downloaded and unpacked.

In [ ]:
found_top = {p.name for p in DATA_DIR.iterdir() if not p.name.startswith(".")}
print("Data folder contains:", sorted(found_top))

add_check(
    "Items of the README folder structure exist",
    f"{CARD}, Folder Structure",
    True,
    {"fake_songs", "real_songs.csv", "fake_songs.csv"} <= found_top,
)
add_check(
    "Files in datasets/05_sonics/",
    f"{WEB}, Files and versions",
    {
        "README.md",
        "fake_songs",
        "fake_songs.csv",
        "real_songs.csv",
        "train.csv",
        "valid.csv",
        "test.csv",
        "metadata.json",
    },  # fmt: skip
    found_top,
)
add_check(
    "Archives in fake_songs/",
    f"{WEB}, folder fake_songs",
    {f"part_{i:02d}.zip" for i in range(1, 11)},
    {p.name for p in AUDIO_DIR.glob("*.zip")},
)
audio_suffixes = {
    p.suffix
    for p in AUDIO_DIR.rglob("*")
    if p.is_file() and not p.name.startswith(".") and p.suffix != ".zip"
}
add_check(
    "Extension of the audio files", f"{CARD}, Folder Structure (yyy.mp3)", {".mp3"}, audio_suffixes
)

### Step 2 – Load the metadata files

**What it does:** Reads `fake_songs.csv`, `real_songs.csv` and the split files `train.csv`, `valid.csv`, `test.csv` into tables. All values are read as text. The files contain the song lyrics with line breaks inside quoted cells, so one row can take many text lines – pandas reads them correctly.

**Why:** All the next checks use these tables. The labels are in `label` (`full fake`, `half fake`, `mostly fake`, `real`) and `target` (`1` fake, `0` real).

In [ ]:
fake = pd.read_csv(FAKE_CSV, dtype=str, keep_default_na=False)
real = pd.read_csv(REAL_CSV, dtype=str, keep_default_na=False)
split_files = {
    split: pd.read_csv(DATA_DIR / f"{split}.csv", dtype=str, keep_default_na=False)
    for split in SPLITS
}
fake["generator"] = fake["algorithm"].map(GENERATORS)

print("Rows in fake_songs.csv:", len(fake))
print("Rows in real_songs.csv:", len(real))
for split, table in split_files.items():
    print(f"Rows in {split}.csv:", len(table))
display(fake.drop(columns=["lyrics", "lyrics_feature"]).head())
display(real.drop(columns=["lyrics"]).head())

### Step 3 – Format of the metadata files

**What it does:**

- Compares the columns of `fake_songs.csv` and `real_songs.csv` with the column tables of `README.md` ("Metadata Properties"). Known differences are kept as DIFF.
- Checks the values of `label`, `target`, `split`, `source` and `no_vocal`. Label and source words are compared without upper / lower case (the paper writes *"Full Fake"*, the file `full fake`). The generator names are compared with the paper (p. 4): the files use other words (`chirp-v3.5` instead of Suno v3.5) – this is kept as a DIFF.
- Checks duplicated file names, duplicated `id` values (`README.md`: *"Unique file ID"*), duplicated YouTube IDs and empty cells in the key columns.

**Why:** If the metadata were broken or used other values than documented, every later result would be wrong.

In [ ]:
add_check(
    "fake_songs.csv: columns",
    f"{CARD}, fake_songs.csv table",
    set(CARD_FAKE_COLUMNS),
    set(fake.columns) - {"generator"},
)
add_check(
    "real_songs.csv: columns",
    f"{CARD}, real_songs.csv table",
    set(CARD_REAL_COLUMNS),
    set(real.columns),
)
add_check(
    "train / valid / test.csv: columns",
    NOT_STATED,
    None,
    sorted({column for table in split_files.values() for column in table.columns}),
)

add_check(
    "fake: label values", f"{CARD}, fake_songs.csv table", set(FAKE_TYPES), set(fake["label"])
)
add_check(
    "fake: label values, ignoring case",
    f"{PAPER}, p. 4, Table 3",
    {"full fake", "half fake", "mostly fake"},
    set(fake["label"].str.lower()),
)
add_check("real: label values", f"{CARD}, real_songs.csv table", {"real"}, set(real["label"]))
add_check("fake: target values", f"{CARD}, fake_songs.csv table", {"1"}, set(fake["target"]))
add_check("real: target values", f"{CARD}, real_songs.csv table", {"0"}, set(real["target"]))
add_check(
    "split values (fake and real)",
    f"{CARD} (train/test/valid split)",
    set(SPLITS),
    set(fake["split"]) | set(real["split"]),
)
add_check(
    "fake: source values, ignoring case",
    f"{CARD} (Suno or Udio)",
    {"suno", "udio"},
    set(fake["source"].str.lower()),
)
add_check(
    "fake: generator names (algorithm) vs paper",
    f"{PAPER}, p. 4, section 3.1.2",
    set(GENERATORS.values()),
    set(fake["algorithm"]),
)
add_check(
    "no_vocal values (fake and real)",
    f"{PAPER}, p. 4 (songs without vocals removed); {CARD}",
    {"False"},
    set(fake["no_vocal"]) | set(real["no_vocal"]),
)

add_check("fake: duplicated filename", "-", 0, int(fake["filename"].duplicated().sum()))
add_check("real: duplicated filename", "-", 0, int(real["filename"].duplicated().sum()))
add_check("fake: duplicated id", f"{CARD} (Unique file ID)", 0, int(fake["id"].duplicated().sum()))
add_check("real: duplicated youtube_id", "-", 0, int(real["youtube_id"].duplicated().sum()))

KEY_COLUMNS = ["filename", "label", "target", "split", "duration"]
empty_key = int((fake[KEY_COLUMNS] == "").sum().sum() + (real[KEY_COLUMNS] == "").sum().sum())
add_check("Empty cells in filename / label / target / split / duration", "-", 0, empty_key)

empty_cells = (
    pd.DataFrame({"fake_songs.csv": (fake == "").sum(), "real_songs.csv": (real == "").sum()})
    .fillna(0)
    .astype(int)
)
empty_cells = empty_cells[empty_cells.sum(axis=1) > 0]
display(empty_cells)

### Step 4 – Number of songs per label

**What it does:** Counts the fake and real songs and the three fake types and compares them with the paper: *"97,164 songs (4,751 hours), including 49,074 end-to-end synthetic songs (1,971 hours) … alongside 48,090 real songs (2,780 hours)"* (p. 2) and *"6,132 HF songs, 40,769 MF songs, and 2,173 FF songs"* (p. 5).

**Why:** Confirms that the metadata describe the whole dataset of the paper.

In [ ]:
label_counts = (
    pd.concat([fake["label"], real["label"]]).value_counts().reindex([*FAKE_TYPES, "real"])
)
counts = label_counts.to_frame("songs")
counts.loc["all"] = int(label_counts.sum())
display(counts)

add_check(
    "fake / real / all songs",
    f"{PAPER}, p. 2, section 1",
    (49074, 48090, 97164),
    (len(fake), len(real), len(fake) + len(real)),
)
add_check(
    "full / half / mostly fake songs",
    f"{PAPER}, p. 5, section 3.1.2",
    (2173, 6132, 40769),
    tuple(int(label_counts[label]) for label in FAKE_TYPES),
)
add_check(
    "Share of fake songs [%]",
    NOT_STATED,
    None,
    round(100 * len(fake) / (len(fake) + len(real)), 2),
)

### Step 5 – Artists

**What it does:** Counts the different artists of the real songs and compares the number with the paper: *"This process yielded 48,090 songs performed by 9,096 artists."* (p. 4). The fake songs have no artist column. It shows the number of songs per artist (the ten largest and the minimum, median and maximum) and the number of artists in each split.

**Why:** The paper uses *"unseen singers"* in valid and test (step 6). If a few artists had very many songs, the artist could become a shortcut.

In [ ]:
add_check(
    "Artists of the real songs", f"{PAPER}, p. 4, section 3.1.1", 9096, real["artist"].nunique()
)

songs_per_artist = real["artist"].value_counts()
display(songs_per_artist.head(10).to_frame("real songs"))
add_check(
    "Real songs per artist: min / median / max",
    NOT_STATED,
    None,
    (
        int(songs_per_artist.min()),
        float(songs_per_artist.median()),
        int(songs_per_artist.max()),
    ),
)
artists_per_split = real.groupby("split")["artist"].nunique().reindex(SPLITS)
add_check(
    "Artists per split (train, valid, test)", NOT_STATED, None, tuple(artists_per_split.tolist())
)

### Step 6 – Splits

**What it does:**

- Counts the songs in `train.csv`, `valid.csv` and `test.csv` and the real songs per split, and compares them with Table 3 of the paper (p. 4).
- Compares all cells of Table 3 (fake type × generator × split) with the counts in `fake_songs.csv` (the number of cells that differ should be 0).
- Checks that train contains only Suno v3.5 and Udio 130 and that the *"unseen algorithms (e.g., Suno v2, Suno v3, Udio 32)"* appear only in valid and test (p. 7).
- Checks that the column `split` of `fake_songs.csv` / `real_songs.csv` agrees with the split files, that no `id` (one lyrics-and-style input) is both in train and in valid / test – *"song pairs from the same (lyrics, style) inputs are exclusively in either the training or valid-test sets"* (p. 7) – and that valid and test contain artists that are not in train (*"unseen singers"*, p. 7).

**Why:** The splits decide which songs a detector has seen in training.

In [ ]:
for split in SPLITS:
    add_check(
        f"{split}.csv: songs",
        f"{PAPER}, p. 4, Table 3",
        TABLE3_SPLIT_TOTALS[split],
        len(split_files[split]),
    )
real_per_split = real["split"].value_counts()
add_check(
    "Real songs per split (train, valid, test)",
    f"{PAPER}, p. 4, Table 3",
    tuple(TABLE3_REAL[split] for split in SPLITS),
    tuple(int(real_per_split.get(split, 0)) for split in SPLITS),
)

table3_measured = pd.crosstab([fake["label"], fake["generator"]], fake["split"]).reindex(
    index=TABLE3.index, columns=SPLITS, fill_value=0
)
display(table3_measured)
add_check(
    "Table 3: cells that differ (fake type x generator x split)",
    f"{PAPER}, p. 4, Table 3; generator names: {NOTE}, section 7",
    0,
    int((table3_measured != TABLE3).sum().sum()),
)

generators_per_split = pd.crosstab(fake["generator"], fake["split"]).reindex(columns=SPLITS)
display(generators_per_split)
add_check(
    "Generators in train",
    f"{PAPER}, p. 7, section 4.1; Table 3",
    {"Suno v3.5", "Udio 130"},
    set(fake.loc[fake["split"] == "train", "generator"]),
)
add_check(
    "Unseen generators only in valid / test",
    f"{PAPER}, p. 7, section 4.1",
    True,
    bool(set(fake.loc[fake["generator"].isin(UNSEEN), "split"]) <= {"valid", "test"}),
)

split_of = pd.concat(
    [table[["filename"]].assign(split_file=split) for split, table in split_files.items()]
).set_index("filename")["split_file"]
both = pd.concat([fake[["filename", "split"]], real[["filename", "split"]]])
add_check(
    "Column split = split file of the song",
    "-",
    True,
    bool((both["split"].to_numpy() == split_of.reindex(both["filename"]).to_numpy()).all()),
)

id_splits = fake.groupby("id")["split"].agg(set)
add_check(
    "ids in train and in valid / test at the same time",
    f"{PAPER}, p. 7, section 4.1",
    0,
    int(id_splits.map(lambda s: "train" in s and len(s) > 1).sum()),
)
train_artists = set(real.loc[real["split"] == "train", "artist"])
add_check(
    "valid / test contain artists that are not in train",
    f"{PAPER}, p. 7, section 4.1 (unseen singers)",
    True,
    bool(set(real.loc[real["split"] != "train", "artist"]) - train_artists != set()),
)

fake_share = (
    fake["split"].value_counts() / (fake["split"].value_counts() + real_per_split) * 100
).reindex(SPLITS)
add_check(
    "Share of fake songs per split [%]", NOT_STATED, None, tuple(fake_share.round(1).tolist())
)

### Step 7 – Audio files vs metadata

**What it does:** Collects all `.mp3` files under `fake_songs/` and compares them with `fake_songs.csv` (songs without a file, files without a row, the same file name in more than one folder) and with `metadata.json`, which lists the archive of every file. It also shows how many files are in each folder. The real songs are not distributed (`README.md`: *"This dataset contains only fake songs"*), so their audio is only counted as an observation.

**Why:** Every fake song must exist as a file, and every file must have a row in the metadata.

In [ ]:
mp3_files = sorted(AUDIO_DIR.rglob("*.mp3"))
mp3_paths = {p.stem: p for p in mp3_files}
on_disk = set(mp3_paths)
in_metadata = set(fake["filename"])

print(f"Fake audio files on disk = {len(mp3_files)}, rows in fake_songs.csv = {len(fake)}")
add_check("Fake songs without an audio file", "-", 0, len(in_metadata - on_disk))
add_check("Audio files without a row in fake_songs.csv", "-", 0, len(on_disk - in_metadata))
add_check("Same file name in more than one folder", "-", 0, len(mp3_files) - len(mp3_paths))

file_mapping = pd.read_json(METADATA_JSON, typ="series")["file_mapping"]
mapped = {Path(name).stem for name in file_mapping}
add_check(
    "Audio files vs metadata.json (missing + extra)",
    "metadata.json (file_mapping)",
    0,
    len(mapped - on_disk) + len(on_disk - mapped),
)

files_per_folder = (
    pd.Series([p.parent.name for p in mp3_files]).value_counts().sort_index().to_frame("mp3 files")
)
files_per_folder.loc["all"] = int(files_per_folder["mp3 files"].sum())
display(files_per_folder)

real_audio = list((DATA_DIR / "real_songs").glob("*")) if (DATA_DIR / "real_songs").exists() else []
add_check(
    "Real-song audio files on disk (real_songs/)",
    f"{CARD} (only fake songs)",
    None,
    len(real_audio),
)

### Step 8a – Read the audio file properties

**What it does:** Reads only the header of every `.mp3` file (not the sound itself): format, sample rate, subtype, number of channels and length. A file that cannot be read is recorded with its error message instead of stopping the notebook. The result is saved to `outputs/05_sonics_dataset_exploration_output/audio_info.csv`; the next time the notebook loads this file instead of reading all files again. **The first run takes about one minute (49,074 files).**

**Why:** These properties are needed for the technical checks in the next cell and later for the table of recording features (silence, loudness, bandwidth, duration).

In [ ]:
INFO_CACHE = OUTPUTS_DIR / "audio_info.csv"

if INFO_CACHE.exists():
    info = pd.read_csv(INFO_CACHE, dtype={"filename": str}, keep_default_na=False)
    print("Loaded from cache:", INFO_CACHE.relative_to(REPO_ROOT))
else:
    rows = []
    paths = [mp3_paths[name] for name in sorted(in_metadata & on_disk)]
    print(f"Reading {len(paths)} files ...")
    for path in paths:
        row = {"filename": path.stem, "folder": path.parent.name, "error": ""}
        try:
            header = sf.info(path)
            row.update(
                {
                    "format": header.format,
                    "subtype": header.subtype,
                    "samplerate": header.samplerate,
                    "channels": header.channels,
                    "frames": header.frames,
                }
            )
        except Exception as error:  # a file that cannot be read is recorded, not a crash
            row["error"] = str(error)[:200]
        rows.append(row)
    info = pd.DataFrame(rows)
    info["duration_s"] = info["frames"] / info["samplerate"]
    info.to_csv(INFO_CACHE, index=False)
    print("Saved to:", INFO_CACHE.relative_to(REPO_ROOT))

audio = info.merge(fake, on="filename", how="inner")
audio["duration_csv"] = audio["duration"].astype(float)
print("Audio files with metadata:", len(audio))

### Step 8b – Check the audio file properties

**What it does:**

- Checks that the files are MP3 (`README.md`) and that `soundfile` could read every header. Sample rate, subtype and channels are not documented – they are only observations; the table shows every value found and the number of files.
- Adds up the hours and compares them with the paper (p. 2: 1,971 h fake, 2,780 h real, 4,751 h in total; rounded to whole hours as in the paper) – once from the **audio headers** (fake songs only) and once from the **`duration` column** of the metadata (fake and real songs). The average length is compared with Table 1 (176.03 s, p. 2).
- Compares the length from the header with the `duration` column of `fake_songs.csv` (`README.md`: *"Total duration (seconds)"*). A difference of more than 0.5 s counts as a difference – this limit is not from a source.
- Compares the longest song of each generator (rounded to 1 s) with the maximum length in the paper (p. 4: Suno v2 80 s, Suno v3 120 s, Suno v3.5 240 s, Udio 32 and 130 s) and the range of all songs with *"from 32 to 240 sec"* (p. 3).
- Shows duration statistics per generator (headers) and for the real songs (metadata).

**Why:** Confirms the technical description of the dataset. A clear difference in duration between generators or between real and fake songs could be a shortcut that detectors use.

In [ ]:
read_ok = audio[audio["error"] == ""]
add_check(
    "File format", f"{CARD}, Folder Structure (yyy.mp3)", {"MP3"}, set(read_ok["format"].tolist())
)
add_check("Files that soundfile.info cannot read", "-", 0, int((audio["error"] != "").sum()))
add_check("Sample rate [Hz]", NOT_STATED, None, set(read_ok["samplerate"].tolist()))
add_check("Subtype", NOT_STATED, None, set(read_ok["subtype"].tolist()))
add_check("Number of channels", NOT_STATED, None, set(read_ok["channels"].tolist()))

PROPERTIES = ["format", "subtype", "samplerate", "channels"]
property_values = pd.concat(
    [
        read_ok[column]
        .value_counts()
        .rename_axis("value")
        .reset_index(name="files")
        .assign(property=column)
        for column in PROPERTIES
    ]
).set_index(["property", "value"])
display(property_values)

fake_hours_header = read_ok["duration_s"].sum() / 3600
fake_hours_csv = fake["duration"].astype(float).sum() / 3600
real_hours_csv = real["duration"].astype(float).sum() / 3600
hours_table = pd.DataFrame(
    {
        "audio headers": [fake_hours_header, np.nan, np.nan],
        "duration column": [fake_hours_csv, real_hours_csv, fake_hours_csv + real_hours_csv],
    },
    index=["fake", "real", "all"],
)
display(hours_table.round(1))

add_check("fake: hours (audio headers)", f"{PAPER}, p. 2", 1971, round(fake_hours_header))
add_check("fake: hours (duration column)", f"{PAPER}, p. 2", 1971, round(fake_hours_csv))
add_check("real: hours (duration column)", f"{PAPER}, p. 2", 2780, round(real_hours_csv))
add_check(
    "all: hours (duration column)", f"{PAPER}, p. 2", 4751, round(fake_hours_csv + real_hours_csv)
)
all_durations = pd.concat([fake["duration"], real["duration"]]).astype(float)
add_check(
    "Average length [s] (duration column)",
    f"{PAPER}, p. 2, Table 1",
    176.03,
    round(float(all_durations.mean()), 2),
)

difference = read_ok["duration_s"] - read_ok["duration_csv"]
add_check(
    "Files where header and duration column differ by more than 0.5 s",
    f"{CARD} (duration: Total duration (seconds)); limit 0.5 s not from a source",
    0,
    int((difference.abs() > 0.5).sum()),
)
add_check(
    "Header minus duration column [s]: min / median / max",
    NOT_STATED,
    None,
    (
        round(float(difference.min()), 3),
        round(float(difference.median()), 3),
        round(float(difference.max()), 3),
    ),
)

longest = read_ok.groupby("generator")["duration_s"].max()
for generator, limit in GENERATOR_LIMITS.items():
    add_check(
        f"{generator}: longest song {longest[generator]:.2f} s (rounded) <= {limit} s",
        f"{PAPER}, p. 4, section 3.1.2",
        True,
        bool(round(longest[generator]) <= limit),
    )
real_durations = real["duration"].astype(float)
add_check(
    "fake songs (headers) between 32 and 240 s",
    f"{PAPER}, p. 3, section 3.1",
    True,
    bool(read_ok["duration_s"].round().between(32, 240).all()),
)
add_check(
    "real songs (duration column) between 32 and 240 s",
    f"{PAPER}, p. 3, section 3.1",
    True,
    bool(real_durations.round().between(32, 240).all()),
)

duration_stats = read_ok.groupby("generator")["duration_s"].describe().reindex(GENERATOR_LIMITS)
duration_stats.loc["real (duration column)"] = real_durations.describe()
duration_stats["count"] = duration_stats["count"].astype(int)
display(duration_stats.round(2))
add_check(
    "Median length [s]: fake (headers) / real (duration column)",
    NOT_STATED,
    None,
    (round(float(read_ok["duration_s"].median()), 2), round(float(real_durations.median()), 2)),
)

### Step 8c – Duration histograms

**What it does:** Draws the same three charts as for the other datasets (density, number of songs, percent) twice:

1. fake songs per generator, with the lengths from the **audio headers**;
2. real vs fake songs, with the lengths from the **`duration` column** of the metadata – the real songs have no audio on disk.

**Why:** Makes it easy to see at a glance whether the generators and the real songs differ in length.

In [ ]:
def plot_durations(data: pd.DataFrame, column: str, group: str, name: str) -> None:
    bins = np.linspace(0, float(np.ceil(data[column].max())), 101)
    variants = [
        ("density", "Density", ""),
        ("counts", "Number of songs", " (counts)"),
        ("percent", "Share of songs in the group [%]", " (percent)"),
    ]
    for variant, y_label, title_end in variants:
        fig, ax = plt.subplots(figsize=(9, 4))
        for label, part in data.groupby(group):
            weights = [100 / len(part)] * len(part) if variant == "percent" else None
            ax.hist(
                part[column],
                bins=bins,
                alpha=0.5,
                density=variant == "density",
                weights=weights,
                label=label,
            )
        ax.set_xlabel("Duration [s]")
        ax.set_ylabel(y_label)
        ax.set_title(f"SONICS, {name}{title_end}")
        ax.legend()
        plt.show()


plot_durations(read_ok, "duration_s", "generator", "fake songs per generator (audio headers)")

by_label = pd.concat(
    [
        fake[["duration"]].assign(group="fake"),
        real[["duration"]].assign(group="real"),
    ]
)
by_label["duration_csv"] = by_label["duration"].astype(float)
plot_durations(by_label, "duration_csv", "group", "real vs fake (duration column of the metadata)")

### Step 9 – Summary of all checks

**What it does:** Shows all checks in one table with their status (OK, DIFF, INFO), lists only the differences and saves the table to `outputs/05_sonics_dataset_exploration_output/checks.csv`.

**Why:** One place to see what matches the documentation and what does not.

In [ ]:
summary = pd.DataFrame(checks)
summary["status"] = summary["ok"].map({True: "OK", False: "DIFF"}).fillna("INFO")
summary = summary[["status", "check", "expected", "measured", "source"]]
display(summary)

summary.to_csv(OUTPUTS_DIR / "checks.csv", index=False)
print("Saved to:", OUTPUTS_DIR.relative_to(REPO_ROOT) / "checks.csv")

differences = summary[summary["status"] == "DIFF"]
print("\nDifferences:")
display(differences)

### Step 10 – Report

**What it does:** Writes the results of all steps (tables only, no charts) to `results/05_sonics/05_sonics_dataset_exploration_output/dataset_exploration_report.md`. Sets (e.g. column names) are written in full.

**Why:** The results can be read later without running the notebook again.

In [ ]:
REPORT_PATH = RESULTS_DIR / "dataset_exploration_report.md"


def fmt(value) -> str:
    if isinstance(value, set):  # written in full, sorted
        if not value:
            return "{} (empty)"
        return "{" + ", ".join(sorted(map(str, value))) + "}"
    if isinstance(value, float):
        return f"{value:.2f}"
    return str(value)


def df_to_md(df: pd.DataFrame, index: bool = True) -> str:
    data = df.reset_index() if index else df
    header = "| " + " | ".join(map(str, data.columns)) + " |"
    separator = "|" + "---|" * len(data.columns)
    rows = ["| " + " | ".join(fmt(v) for v in row) + " |" for row in data.itertuples(index=False)]
    return "\n".join([header, separator, *rows])


status_counts = summary["status"].value_counts()

sections = [
    ("Step 3 – Empty cells per column", empty_cells),
    ("Step 4 – Number of songs per label", counts),
    ("Step 5 – Ten artists with the most real songs", songs_per_artist.head(10).to_frame("songs")),
    ("Step 6 – Fake songs per fake type, generator and split (Table 3)", table3_measured),
    ("Step 6 – Fake songs per generator and split", generators_per_split),
    ("Step 7 – Audio files per folder", files_per_folder),
    ("Step 8 – Audio file properties", property_values),
    ("Step 8 – Hours", hours_table.round(1)),
    ("Step 8 – Duration of songs [s]", duration_stats.round(2)),
]

lines = [
    "# SONICS – dataset check report",
    "",
    f"- Generated: {datetime.now():%Y-%m-%d %H:%M}",
    f"- Dataset folder: `{DATA_DIR.relative_to(REPO_ROOT)}`",
    f"- Notebook: `notebooks/{NOTEBOOK_NAME}.ipynb`",
    "- Only fake songs are distributed as audio; real songs are checked with metadata only.",
    "",
    "## Overview",
    "",
    f"- Checks in total: {len(summary)}",
    f"- OK (matches the documentation): {status_counts.get('OK', 0)}",
    f"- DIFF (does not match): {status_counts.get('DIFF', 0)}",
    f"- INFO (observation, nothing to compare with): {status_counts.get('INFO', 0)}",
    "",
    "## Differences",
    "",
    df_to_md(differences, index=False) if len(differences) else "No differences found.",
    "",
    "## All checks",
    "",
    df_to_md(summary, index=False),
    "",
]
for title, table in sections:
    lines += [f"## {title}", "", df_to_md(table), ""]

REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print("Report saved to:", REPORT_PATH.relative_to(REPO_ROOT))